In [7]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score, roc_curve)

pd.options.display.max_columns = None
sns.set_style('whitegrid')


In [8]:
df = pd.read_csv("ibm hr dataset.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [10]:
df.columns

Index(['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department',
       'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount',
       'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction',
       'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating',
       'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager'],
      dtype='object')

In [11]:
df.shape

(1470, 35)

In [12]:
df.isnull().sum().sum()

np.int64(0)

#Exploratory Analysis

**Q1. Does age affect attrition?**

In [13]:
age_att = df.groupby(['Age','Attrition']).size().reset_index(name='Counts')
px.line(age_att, x='Age', y='Counts', color='Attrition',
        title='Attrition Counts by Age')

*Takeaway - Attrition (red line) peaks among employees in their late 20s to early 30s, then steadily declines as age increases, younger employees are more likely to leave than older, more settled ones.*

**Q2. Is pay the main driver of attrition?**

In [14]:
rate_att = df.groupby(['MonthlyIncome','Attrition']).size().reset_index(name='Counts')
rate_att['MonthlyIncome'] = round(rate_att['MonthlyIncome'], -3)
rate_att = rate_att.groupby(['MonthlyIncome','Attrition']).sum(numeric_only=True).reset_index()
px.line(rate_att, x='MonthlyIncome', y='Counts', color='Attrition',
        title='Attrition Counts by Monthly Income (rounded to nearest $1k)')

**Takeaway:** Attrition is heavily concentrated below the ~$5k/month mark and flattens out at higher pay. This says pay matters most as a *floor*, it stops people from leaving out of financial necessity, but past a certain point it's not the thing keeping people engaged.

**Q3. Does department matter?**

In [15]:
dept_att = df.groupby(['Department','Attrition']).size().reset_index(name='Counts')
dept_totals = df.groupby('Department').size()
dept_att['Rate'] = dept_att.apply(lambda r: r['Counts']/dept_totals[r['Department']], axis=1)
px.bar(dept_att, x='Department', y='Counts', color='Attrition',
       title='Attrition Counts by Department')


**Takeaway:** Sales has the highest attrition rate, followed by HR, with R&D the most stable. Worth flagging that Sales roles are often externally poachable (commission-based comp is portable to a competitor), which is a different retention problem than an R&D role would have.

**Q4. Is OverTime a bigger red flag than job satisfaction? (new question)
The original analysis didn't touch OverTime, but it turns out to be one of the strongest signals in this dataset - worth checking before anything else.**

In [16]:
ot_att = df.groupby(['OverTime','Attrition']).size().reset_index(name='Counts')
ot_totals = df.groupby('OverTime').size()
ot_att['Rate'] = ot_att.apply(lambda r: r['Counts']/ot_totals[r['OverTime']], axis=1)
fig = px.bar(ot_att, x='OverTime', y='Rate', color='Attrition',
             title='Attrition Rate by OverTime Status', barmode='group')
fig.show()
ot_att


,OverTime,Attrition,Counts,Rate
0,No,No,944,0.895636
1,No,Yes,110,0.104364
2,Yes,No,289,0.694712
3,Yes,Yes,127,0.305288


**Takeaway:** Employees working overtime leave at a noticeably higher rate than those who don't. This is arguably more actionable than "increase satisfaction scores" — it's a workload/staffing decision, not a vague culture initiative.`

**Q5. Satisfaction (environment & job)**

In [17]:
sats_att = df.groupby(['EnvironmentSatisfaction','Attrition']).size().reset_index(name='Counts')
px.area(sats_att, x='EnvironmentSatisfaction', y='Counts', color='Attrition',
        title='Attrition by Environment Satisfaction')


In [18]:
jsats_att = df.groupby(['JobSatisfaction','Attrition']).size().reset_index(name='Counts')
px.area(jsats_att, x='JobSatisfaction', y='Counts', color='Attrition',
        title='Attrition by Job Satisfaction')

**Takeaway:** Both satisfaction measures show the expected direction (lower satisfaction, higher attrition), though the relationship isn't perfectly linear - there's a small bump in the middle satisfaction levels, which usually means "comfortable enough to look elsewhere" rather than "miserable and desperate to leave."

**Q6. Stock options and work-life balance**

In [19]:
stock_att = df.groupby(['StockOptionLevel','Attrition']).size().reset_index(name='Counts')
px.bar(stock_att, x='StockOptionLevel', y='Counts', color='Attrition',
       title='Attrition by Stock Option Level')


In [20]:
wlb_att = df.groupby(['WorkLifeBalance','Attrition']).size().reset_index(name='Counts')
px.bar(wlb_att, x='WorkLifeBalance', y='Counts', color='Attrition',
       title='Attrition by Work-Life Balance Rating')


**Takeaway:** No/low stock options line up with higher attrition — unvested equity is a real "golden handcuff." Work-life balance shows the highest attrition at the lowest rating, which combined with the OverTime finding above, points at the same underlying issue: workload.

**Q7. Tenure & promotion: are we creating a 'stuck' effect?**

In [21]:
promo_att = df.groupby(['YearsSinceLastPromotion','Attrition']).size().reset_index(name='Counts')
px.line(promo_att, x='YearsSinceLastPromotion', y='Counts', color='Attrition',
        title='Attrition by Years Since Last Promotion')


**Takeaway:** There's a cluster of attrition among people who haven't been promoted in 0-1 years too — which seems counterintuitive until you realize this dataset also has a lot of very short-tenure employees. The more useful signal is the interaction with `YearsAtCompany`: people with long tenure and long promotion gaps are the ones worth watching (checked below in the risk segment).

**Q8. Manager relationship & commute**

In [22]:
man_att = df.groupby(['YearsWithCurrManager','Attrition']).size().reset_index(name='Counts')
px.line(man_att, x='YearsWithCurrManager', y='Counts', color='Attrition',
        title='Attrition by Years With Current Manager')


In [23]:
dist_att = df.groupby(['DistanceFromHome','Attrition']).size().reset_index(name='Counts')
px.line(dist_att, x='DistanceFromHome', y='Counts', color='Attrition',
        title='Attrition by Distance From Home')


**Takeaway:** Attrition spikes early in a manager relationship (still evaluating fit) and again around year 2 (the classic "two-year itch"). Longer commutes also correlate with higher attrition — a cheap, practical lever (hybrid/remote flexibility) that's easy to overlook compared to comp-based fixes.

# The Dollar Question: What Is Attrition Actually Costing Us?
Percentages don't get budget approved — dollar figures do. Using a commonly cited SHRM/industry rule of thumb that replacing an employee costs roughly **50% of their annual salary** (recruiting, onboarding, lost productivity while the role is empty/ramping), here's a rough cost estimate based on this dataset's income figures.

This is obviously a simplification (real replacement cost varies a lot by role seniority), but it's enough to make the point that attrition isn't just an HR metric — it's a line item.

In [27]:
REPLACEMENT_COST_FACTOR = 0.5  # rule-of-thumb: ~50% of annual salary per departure

leavers = df[df['Attrition'] == 'Yes'].copy()
leavers['AnnualIncome'] = leavers['MonthlyIncome'] * 12
leavers['EstReplacementCost'] = leavers['AnnualIncome'] * REPLACEMENT_COST_FACTOR

total_cost = leavers['EstReplacementCost'].sum()
avg_cost = leavers['EstReplacementCost'].mean()

print(f"Employees who left: {len(leavers)}")
print(f"Estimated total replacement cost: ${total_cost:,.0f}")
print(f"Estimated average cost per departure: ${avg_cost:,.0f}")

Employees who left: 237
Estimated total replacement cost: $6,807,246
Estimated average cost per departure: $28,723


In [25]:
cost_by_dept = leavers.groupby('Department')['EstReplacementCost'].sum().sort_values(ascending=False)
px.bar(cost_by_dept, title='Estimated Attrition Cost by Department ($)',
       labels={'value': 'Estimated Cost ($)', 'Department': 'Department'})

**Takeaway:** Even with a conservative estimate, attrition costs add up to a meaningful number here. This reframes the whole exercise: it's not "why do people leave" as an academic question, it's "what's the cheapest lever we can pull to keep the people worth keeping."

In [26]:
import pandas as pd
df = pd.read_csv("ibm hr dataset.csv")
df.to_csv('processed_csv.csv', index=False)